This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## The mathematical building blocks of neural networks

### A first look at a neural network

In [ ]:
# PyTorch: there's no keras.datasets; load MNIST via torchvision and expose the
# raw uint8 NumPy arrays so the shapes/dtypes match what the chapter expects.
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
(train_images, train_labels) = mnist_train.data.numpy(), mnist_train.targets.numpy()
(test_images, test_labels) = mnist_test.data.numpy(), mnist_test.targets.numpy()

In [0]:
train_images.shape

In [0]:
len(train_labels)

In [0]:
train_labels

In [0]:
test_images.shape

In [0]:
len(test_labels)

In [0]:
test_labels

In [ ]:
import torch
from torch import nn

# PyTorch: nn.Sequential needs explicit in/out sizes (no lazy build like Keras),
# and we drop the final softmax because nn.CrossEntropyLoss expects raw logits.
model = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)

In [ ]:
# PyTorch: there is no compile() step. We create the optimizer and loss function
# directly. nn.CrossEntropyLoss combines softmax + sparse categorical crossentropy.
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

In [0]:
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

In [ ]:
# PyTorch: model.fit() becomes an explicit training loop. We turn the NumPy arrays
# into tensors, iterate over shuffled mini-batches, and run the standard
# forward -> loss -> zero_grad -> backward -> step cycle ourselves.
x_train = torch.from_numpy(train_images)
y_train = torch.from_numpy(train_labels).long()

batch_size = 128
for epoch in range(5):
    permutation = torch.randperm(len(x_train))
    for i in range(0, len(x_train), batch_size):
        idx = permutation[i : i + batch_size]
        inputs, targets = x_train[idx], y_train[idx]
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: loss {loss.item():.4f}")

In [ ]:
test_digits = test_images[0:10]
# PyTorch: call the model under torch.no_grad() and apply softmax ourselves to
# turn the logits into probabilities (Keras applied softmax inside the model).
with torch.no_grad():
    predictions = torch.softmax(model(torch.from_numpy(test_digits)), dim=1).numpy()
predictions[0]

In [0]:
predictions[0].argmax()

In [0]:
predictions[0][7]

In [0]:
test_labels[0]

In [ ]:
# PyTorch: replace model.evaluate() with a manual no-grad evaluation pass.
with torch.no_grad():
    predicted_labels = model(torch.from_numpy(test_images)).argmax(dim=1).numpy()
test_acc = (predicted_labels == test_labels).mean()
print(f"test_acc: {test_acc}")

### Data representations for neural networks

#### Scalars (rank-0 tensors)

In [0]:
import numpy as np
x = np.array(12)
x

In [0]:
x.ndim

#### Vectors (rank-1 tensors)

In [0]:
x = np.array([12, 3, 6, 14, 7])
x

In [0]:
x.ndim

#### Matrices (rank-2 tensors)

In [0]:
x = np.array([[5, 78, 2, 34, 0],
              [6, 79, 3, 35, 1],
              [7, 80, 4, 36, 2]])
x.ndim

#### Rank-3 tensors and higher-rank tensors

In [0]:
x = np.array([[[5, 78, 2, 34, 0],
               [6, 79, 3, 35, 1],
               [7, 80, 4, 36, 2]],
              [[5, 78, 2, 34, 0],
               [6, 79, 3, 35, 1],
               [7, 80, 4, 36, 2]],
              [[5, 78, 2, 34, 0],
               [6, 79, 3, 35, 1],
               [7, 80, 4, 36, 2]]])
x.ndim

#### Key attributes

In [ ]:
# PyTorch: there's no keras.datasets; load MNIST via torchvision and expose the
# raw uint8 NumPy arrays so the shapes/dtypes match what the chapter expects.
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
(train_images, train_labels) = mnist_train.data.numpy(), mnist_train.targets.numpy()
(test_images, test_labels) = mnist_test.data.numpy(), mnist_test.targets.numpy()

In [0]:
train_images.ndim

In [0]:
train_images.shape

In [0]:
train_images.dtype

In [0]:
import matplotlib.pyplot as plt

digit = train_images[4]
plt.imshow(digit, cmap=plt.cm.binary)
plt.show()

In [0]:
train_labels[4]

#### Manipulating tensors in NumPy

In [0]:
my_slice = train_images[10:100]
my_slice.shape

In [0]:
my_slice = train_images[10:100, :, :]
my_slice.shape

In [0]:
my_slice = train_images[10:100, 0:28, 0:28]
my_slice.shape

In [0]:
my_slice = train_images[:, 14:, 14:]

In [0]:
my_slice = train_images[:, 7:-7, 7:-7]

#### The notion of data batches

In [0]:
batch = train_images[:128]

In [0]:
batch = train_images[128:256]

In [0]:
n = 3
batch = train_images[128 * n : 128 * (n + 1)]

#### Real-world examples of data tensors

##### Vector data

##### Timeseries data or sequence data

##### Image data

##### Video data

### The gears of neural networks: Tensor operations

#### Element-wise operations

In [0]:
def naive_relu(x):
    assert len(x.shape) == 2
    x = x.copy()
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            x[i, j] = max(x[i, j], 0)
    return x

In [0]:
def naive_add(x, y):
    assert len(x.shape) == 2
    assert x.shape == y.shape
    x = x.copy()
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            x[i, j] += y[i, j]
    return x

In [0]:
import time

x = np.random.random((20, 100))
y = np.random.random((20, 100))

t0 = time.time()
for _ in range(1000):
    z = x + y
    z = np.maximum(z, 0.0)
print("Took: {0:.2f} s".format(time.time() - t0))

In [0]:
t0 = time.time()
for _ in range(1000):
    z = naive_add(x, y)
    z = naive_relu(z)
print("Took: {0:.2f} s".format(time.time() - t0))

#### Broadcasting

In [0]:
import numpy as np

X = np.random.random((32, 10))
y = np.random.random((10,))

In [0]:
y = np.expand_dims(y, axis=0)

In [0]:
Y = np.tile(y, (32, 1))

In [0]:
def naive_add_matrix_and_vector(x, y):
    assert len(x.shape) == 2
    assert len(y.shape) == 1
    assert x.shape[1] == y.shape[0]
    x = x.copy()
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            x[i, j] += y[j]
    return x

In [0]:
import numpy as np

x = np.random.random((64, 3, 32, 10))
y = np.random.random((32, 10))
z = np.maximum(x, y)

#### Tensor product

In [0]:
x = np.random.random((32,))
y = np.random.random((32,))

z = np.matmul(x, y)
z = x @ y

In [0]:
def naive_vector_product(x, y):
    assert len(x.shape) == 1
    assert len(y.shape) == 1
    assert x.shape[0] == y.shape[0]
    z = 0.0
    for i in range(x.shape[0]):
        z += x[i] * y[i]
    return z

In [0]:
def naive_matrix_vector_product(x, y):
    assert len(x.shape) == 2
    assert len(y.shape) == 1
    assert x.shape[1] == y.shape[0]
    z = np.zeros(x.shape[0])
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            z[i] += x[i, j] * y[j]
    return z

In [0]:
def naive_matrix_vector_product(x, y):
    z = np.zeros(x.shape[0])
    for i in range(x.shape[0]):
        z[i] = naive_vector_product(x[i, :], y)
    return z

In [0]:
def naive_matrix_product(x, y):
    assert len(x.shape) == 2
    assert len(y.shape) == 2
    assert x.shape[1] == y.shape[0]
    z = np.zeros((x.shape[0], y.shape[1]))
    for i in range(x.shape[0]):
        for j in range(y.shape[1]):
            row_x = x[i, :]
            column_y = y[:, j]
            z[i, j] = naive_vector_product(row_x, column_y)
    return z

#### Tensor reshaping

In [0]:
train_images = train_images.reshape((60000, 28 * 28))

In [0]:
x = np.array([[0., 1.],
              [2., 3.],
              [4., 5.]])
x.shape

In [0]:
x = x.reshape((6, 1))
x

In [0]:
x = x.reshape((2, 3))
x

In [0]:
x = np.zeros((300, 20))
x = np.transpose(x)
x.shape

#### Geometric interpretation of tensor operations

#### A geometric interpretation of deep learning

### The engine of neural networks: Gradient-based optimization

#### What's a derivative?

#### Derivative of a tensor operation: The gradient

#### Stochastic gradient descent

#### Chaining derivatives: The Backpropagation algorithm

##### The chain rule

##### Automatic differentiation with computation graphs

### Looking back at our first example

In [ ]:
mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
(train_images, train_labels) = mnist_train.data.numpy(), mnist_train.targets.numpy()
(test_images, test_labels) = mnist_test.data.numpy(), mnist_test.targets.numpy()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

In [ ]:
model = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

In [ ]:
x_train = torch.from_numpy(train_images)
y_train = torch.from_numpy(train_labels).long()

batch_size = 128
for epoch in range(5):
    permutation = torch.randperm(len(x_train))
    for i in range(0, len(x_train), batch_size):
        idx = permutation[i : i + batch_size]
        inputs, targets = x_train[idx], y_train[idx]
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

#### Reimplementing our first example from scratch

##### A simple Dense class

In [ ]:
import torch

class NaiveDense:
    def __init__(self, input_size, output_size, activation=None):
        self.activation = activation
        # PyTorch: trainable params are tensors with requires_grad=True. The
        # uniform init mirrors Keras's RandomUniform default of [-0.05, 0.05].
        self.W = torch.empty(input_size, output_size).uniform_(-0.05, 0.05).requires_grad_(True)
        self.b = torch.zeros(output_size, requires_grad=True)

    def __call__(self, inputs):
        x = torch.matmul(inputs, self.W)
        x = x + self.b
        if self.activation is not None:
            x = self.activation(x)
        return x

    @property
    def weights(self):
        return [self.W, self.b]

##### A simple Sequential class

In [0]:
class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights

In [ ]:
# PyTorch: pass torch.relu / torch.softmax as the activation callables.
model = NaiveSequential(
    [
        NaiveDense(input_size=28 * 28, output_size=512, activation=torch.relu),
        NaiveDense(input_size=512, output_size=10, activation=lambda x: torch.softmax(x, dim=-1)),
    ]
)
assert len(model.weights) == 4

##### A batch generator

In [0]:
import math

class BatchGenerator:
    def __init__(self, images, labels, batch_size=128):
        assert len(images) == len(labels)
        self.index = 0
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.num_batches = math.ceil(len(images) / batch_size)

    def next(self):
        images = self.images[self.index : self.index + self.batch_size]
        labels = self.labels[self.index : self.index + self.batch_size]
        self.index += self.batch_size
        return images, labels

#### Running one training step

##### The weight update step

In [ ]:
learning_rate = 1e-3

def update_weights(gradients, weights):
    # PyTorch: update params in-place under no_grad so the op isn't tracked.
    with torch.no_grad():
        for g, w in zip(gradients, weights):
            w -= g * learning_rate

In [ ]:
# PyTorch: use torch.optim.SGD. We assign the precomputed gradients onto each
# parameter's .grad and let the optimizer apply them (like apply_gradients).
optimizer = torch.optim.SGD(model.weights, lr=1e-3)

def update_weights(gradients, weights):
    for g, w in zip(gradients, weights):
        w.grad = g
    optimizer.step()
    optimizer.zero_grad()

##### Gradient computation

In [ ]:
# PyTorch: autograd replaces tf.GradientTape. requires_grad_ tracks operations on
# the tensor, and backward() populates its .grad attribute.
x = torch.zeros(())
x.requires_grad_(True)
y = 2 * x + 3
y.backward()
grad_of_y_wrt_x = x.grad

In [ ]:
import torch.nn.functional as F

def one_training_step(model, images_batch, labels_batch):
    # PyTorch: convert the NumPy batch to tensors before the forward pass.
    images_batch = torch.from_numpy(images_batch)
    labels_batch = torch.from_numpy(labels_batch).long()
    predictions = model(images_batch)
    # Sparse categorical crossentropy from probabilities == NLL of the log-probs.
    loss = F.nll_loss(torch.log(predictions), labels_batch, reduction="none")
    average_loss = loss.mean()
    # PyTorch: torch.autograd.grad returns gradients explicitly, like tape.gradient.
    gradients = torch.autograd.grad(average_loss, model.weights)
    update_weights(gradients, model.weights)
    return average_loss

#### The full training loop

In [ ]:
def fit(model, images, labels, epochs, batch_size=128):
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        batch_generator = BatchGenerator(images, labels)
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            if batch_counter % 100 == 0:
                print(f"loss at batch {batch_counter}: {loss:.2f}")

In [ ]:
mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
(train_images, train_labels) = mnist_train.data.numpy(), mnist_train.targets.numpy()
(test_images, test_labels) = mnist_test.data.numpy(), mnist_test.targets.numpy()

train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

fit(model, train_images, train_labels, epochs=10, batch_size=128)

#### Evaluating the model

In [ ]:
# PyTorch: run the forward pass without tracking gradients for evaluation.
with torch.no_grad():
    predictions = model(torch.from_numpy(test_images))
predicted_labels = predictions.argmax(dim=1).numpy()
matches = predicted_labels == test_labels
f"accuracy: {matches.mean():.2f}"